# Error Metrics Class for Regression and Classification

This notebook implements a reusable `ErrorMetrics` class for common evaluation metrics used in machine learning.

**Metrics covered**
- Mean Absolute Error (MAE)
- Root Mean Square Error (RMSE)
- Log Loss
- R-square (R²)
- Adjusted R-square (Adjusted R²)

## Objective

Build a clean, vectorized Python class that can evaluate model predictions using standard error and goodness-of-fit metrics.

## Theory and Concept Used

These metrics measure how close predictions are to the true values.

- **MAE** measures average absolute deviation.
- **RMSE** penalizes larger errors more strongly.
- **Log Loss** evaluates probabilistic classification output.
- **R²** measures how much variance is explained by a model.
- **Adjusted R²** corrects R² for the number of predictors used.

## Formulas

$$\text{MAE} = \frac{1}{n}\sum_{i=1}^{n}|y_i - \hat{y}_i|$$

$$\text{RMSE} = \sqrt{\frac{1}{n}\sum_{i=1}^{n}(y_i - \hat{y}_i)^2}$$

$$\text{LogLoss} = -\frac{1}{n}\sum_{i=1}^{n}\left[y_i\log(\hat{p}_i) + (1-y_i)\log(1-\hat{p}_i)\right]$$

$$R^2 = 1 - \frac{\sum_i(y_i - \hat{y}_i)^2}{\sum_i(y_i - \bar{y})^2}$$

$$R^2_{adj} = 1 - (1-R^2)\frac{n-1}{n-p-1}$$

## Concepts Used

- NumPy vectorization for fast array computation
- Input validation for shape and finiteness checks
- Numerical stability through probability clipping
- Reusable class-based design
- Test-driven validation with small known examples

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_regression, make_classification
from sklearn.metrics import mean_absolute_error as sk_mean_absolute_error
from sklearn.metrics import mean_squared_error as sk_mean_squared_error
from sklearn.metrics import log_loss as sk_log_loss
from sklearn.metrics import r2_score as sk_r2_score
from sklearn.model_selection import train_test_split
from timeit import timeit
from typing import Iterable

np.random.seed(42)

In [ ]:
# Synthetic data for examples and validation
X_reg, y_reg = make_regression(n_samples=200, n_features=5, noise=12.0, random_state=42)
X_clf, y_clf = make_classification(
    n_samples=200,
    n_features=6,
    n_informative=4,
    n_redundant=0,
    random_state=42,
)

X_reg_train, X_reg_test, y_reg_train, y_reg_test = train_test_split(
    X_reg, y_reg, test_size=0.25, random_state=42
)
X_clf_train, X_clf_test, y_clf_train, y_clf_test = train_test_split(
    X_clf, y_clf, test_size=0.25, random_state=42, stratify=y_clf
)

# Create simple prediction examples without fitting a full model
regression_noise = np.random.normal(loc=0.0, scale=10.0, size=y_reg_test.shape)
y_reg_pred = y_reg_test + regression_noise

classifier_weights = np.array([0.9, -1.2, 0.7, 0.5, -0.6, 1.1])
clf_scores = X_clf_test @ classifier_weights
clf_scores = clf_scores / np.std(clf_scores)
y_prob_pred = 1.0 / (1.0 + np.exp(-clf_scores))

In [ ]:
class ErrorMetrics:
    """Compute common regression and classification evaluation metrics.

    Parameters
    ----------
    eps : float, default=1e-15
        Small constant used to clip probabilities for numerical stability.
    """

    def __init__(self, eps: float = 1e-15) -> None:
        self.eps = float(eps)

    def _to_numpy(self, values: Iterable[float], name: str) -> np.ndarray:
        array = np.asarray(values, dtype=float)
        if array.ndim == 0:
            array = array.reshape(1)
        if array.size == 0:
            raise ValueError(f"{name} cannot be empty")
        if not np.all(np.isfinite(array)):
            raise ValueError(f"{name} must contain only finite values")
        return array

    def _validate_pair(self, y_true: Iterable[float], y_pred: Iterable[float]) -> tuple[np.ndarray, np.ndarray]:
        true_array = self._to_numpy(y_true, "y_true")
        pred_array = self._to_numpy(y_pred, "y_pred")
        if true_array.shape != pred_array.shape:
            raise ValueError("y_true and y_pred must have the same shape")
        return true_array, pred_array

    def mean_absolute_error(self, y_true: Iterable[float], y_pred: Iterable[float]) -> float:
        """Return the mean absolute error between true and predicted values."""
        true_array, pred_array = self._validate_pair(y_true, y_pred)
        return float(np.mean(np.abs(true_array - pred_array)))

    def root_mean_square_error(self, y_true: Iterable[float], y_pred: Iterable[float]) -> float:
        """Return the root mean square error between true and predicted values."""
        true_array, pred_array = self._validate_pair(y_true, y_pred)
        return float(np.sqrt(np.mean((true_array - pred_array) ** 2)))

    def log_loss(self, y_true: Iterable[float], y_prob: Iterable[float]) -> float:
        """Return the binary log loss for 0/1 labels and predicted probabilities."""
        true_array, prob_array = self._validate_pair(y_true, y_prob)
        if not np.isin(true_array, [0.0, 1.0]).all():
            raise ValueError("y_true must contain only binary labels 0 and 1")
        clipped = np.clip(prob_array, self.eps, 1.0 - self.eps)
        loss = -(true_array * np.log(clipped) + (1.0 - true_array) * np.log(1.0 - clipped))
        return float(np.mean(loss))

    def r2_score(self, y_true: Iterable[float], y_pred: Iterable[float]) -> float:
        """Return the coefficient of determination R².

        If the target has zero variance, this method returns NaN because R² is undefined.
        """
        true_array, pred_array = self._validate_pair(y_true, y_pred)
        total_sum_squares = np.sum((true_array - np.mean(true_array)) ** 2)
        if total_sum_squares == 0:
            return float("nan")
        residual_sum_squares = np.sum((true_array - pred_array) ** 2)
        return float(1.0 - (residual_sum_squares / total_sum_squares))

    def adjusted_r2_score(
        self,
        y_true: Iterable[float],
        y_pred: Iterable[float],
        n_features: int,
    ) -> float:
        """Return adjusted R² for n samples and p features.

        Parameters
        ----------
        y_true, y_pred : array-like
            Observed and predicted values.
        n_features : int
            Number of predictors p.
        """
        true_array, pred_array = self._validate_pair(y_true, y_pred)
        n_samples = true_array.size
        if n_features < 0:
            raise ValueError("n_features must be non-negative")
        if n_samples <= n_features + 1:
            raise ValueError("Adjusted R² requires n_samples > n_features + 1")

        r2 = self.r2_score(true_array, pred_array)
        if np.isnan(r2):
            return float("nan")
        return float(1.0 - (1.0 - r2) * ((n_samples - 1) / (n_samples - n_features - 1)))


metrics = ErrorMetrics()

## Validation, Input Checks, and Edge Cases

The class validates:
- matching shapes
- non-empty inputs
- finite values only
- binary labels for log loss
- valid sample count for adjusted R²

Examples below show how the class behaves on edge cases.

In [ ]:
edge_examples = {}

try:
    metrics.mean_absolute_error([1, 2], [1])
except ValueError as exc:
    edge_examples["shape_mismatch"] = str(exc)

try:
    metrics.log_loss([0, 1], [1.2, -0.1])
except ValueError as exc:
    edge_examples["probabilities_outside_range"] = str(exc)

try:
    metrics.adjusted_r2_score([1, 2], [1, 2], n_features=2)
except ValueError as exc:
    edge_examples["too_few_samples"] = str(exc)

edge_examples

## Unit Tests with pytest Style

The following functions are written in `pytest` style. They can be executed by `pytest` or by calling them directly in the notebook.

In [ ]:
def test_mae_known_values():
    assert np.isclose(metrics.mean_absolute_error([1, 2, 3], [2, 2, 4]), 2 / 3)


def test_rmse_known_values():
    assert np.isclose(metrics.root_mean_square_error([1, 2, 3], [2, 2, 4]), np.sqrt(2 / 3))


def test_log_loss_known_values():
    result = metrics.log_loss([0, 1], [0.25, 0.75])
    expected = -0.5 * (np.log(0.75) + np.log(0.75))
    assert np.isclose(result, expected)


def test_r2_known_values():
    result = metrics.r2_score([1, 2, 3], [1, 2, 3])
    assert np.isclose(result, 1.0)


def test_adjusted_r2_known_values():
    result = metrics.adjusted_r2_score([1, 2, 3, 4], [1, 2, 3, 4], n_features=1)
    assert np.isclose(result, 1.0)


def test_shape_mismatch_raises():
    try:
        metrics.mean_absolute_error([1, 2], [1])
        raise AssertionError("Expected ValueError")
    except ValueError:
        pass


for test_function in [
    test_mae_known_values,
    test_rmse_known_values,
    test_log_loss_known_values,
    test_r2_known_values,
    test_adjusted_r2_known_values,
    test_shape_mismatch_raises,
]:
    test_function()

print("All notebook tests passed.")

## Compare with scikit-learn Implementations

The next cell compares the custom implementation against scikit-learn on the same synthetic data.

In [ ]:
custom_mae = metrics.mean_absolute_error(y_reg_test, y_reg_pred)
custom_rmse = metrics.root_mean_square_error(y_reg_test, y_reg_pred)
custom_r2 = metrics.r2_score(y_reg_test, y_reg_pred)
custom_log_loss = metrics.log_loss(y_clf_test, y_prob_pred)
custom_adjusted_r2 = metrics.adjusted_r2_score(y_reg_test, y_reg_pred, n_features=X_reg_test.shape[1])

sklearn_mae = sk_mean_absolute_error(y_reg_test, y_reg_pred)
sklearn_rmse = np.sqrt(sk_mean_squared_error(y_reg_test, y_reg_pred))
sklearn_r2 = sk_r2_score(y_reg_test, y_reg_pred)
sklearn_log_loss = sk_log_loss(y_clf_test, y_prob_pred)
sklearn_adjusted_r2 = 1.0 - (1.0 - sklearn_r2) * ((len(y_reg_test) - 1) / (len(y_reg_test) - X_reg_test.shape[1] - 1))

comparison = pd.DataFrame(
    {
        "metric": ["MAE", "RMSE", "LogLoss", "R2", "Adjusted R2"],
        "custom": [custom_mae, custom_rmse, custom_log_loss, custom_r2, custom_adjusted_r2],
        "sklearn": [sklearn_mae, sklearn_rmse, sklearn_log_loss, sklearn_r2, sklearn_adjusted_r2],
    }
)
comparison["difference"] = (comparison["custom"] - comparison["sklearn"]).abs()
comparison

In [ ]:
np.testing.assert_allclose(custom_mae, sklearn_mae)
np.testing.assert_allclose(custom_rmse, sklearn_rmse)
np.testing.assert_allclose(custom_log_loss, sklearn_log_loss)
np.testing.assert_allclose(custom_r2, sklearn_r2)
np.testing.assert_allclose(custom_adjusted_r2, sklearn_adjusted_r2)
print("Custom metrics match scikit-learn outputs.")

## Performance Benchmarking and Vectorization

This section compares vectorized NumPy implementations against naive Python loops on large arrays. The vectorized version is faster because it avoids explicit Python iteration.

In [ ]:
large_true = np.random.randn(100_000)
large_pred = large_true + np.random.normal(0, 0.5, size=large_true.shape)


def mae_loop(y_true, y_pred):
    total = 0.0
    for actual, predicted in zip(y_true, y_pred):
        total += abs(actual - predicted)
    return total / len(y_true)


def rmse_loop(y_true, y_pred):
    total = 0.0
    for actual, predicted in zip(y_true, y_pred):
        diff = actual - predicted
        total += diff * diff
    return (total / len(y_true)) ** 0.5

vectorized_mae_time = timeit(lambda: metrics.mean_absolute_error(large_true, large_pred), number=10)
loop_mae_time = timeit(lambda: mae_loop(large_true, large_pred), number=1)
vectorized_rmse_time = timeit(lambda: metrics.root_mean_square_error(large_true, large_pred), number=10)
loop_rmse_time = timeit(lambda: rmse_loop(large_true, large_pred), number=1)

benchmark = pd.DataFrame(
    {
        "implementation": ["vectorized MAE", "loop MAE", "vectorized RMSE", "loop RMSE"],
        "seconds": [vectorized_mae_time, loop_mae_time, vectorized_rmse_time, loop_rmse_time],
    }
)
benchmark

## Save Notebook Programmatically

If you want to export notebook content programmatically, `nbformat` can write a notebook object to disk.

In [ ]:
import nbformat
from nbformat.v4 import new_code_cell, new_markdown_cell, new_notebook

notebook_export = new_notebook(
    cells=[
        new_markdown_cell("# Exported notebook example"),
        new_code_cell("print('This is a programmatically generated notebook file.')"),
    ],
    metadata={
        "kernelspec": {
            "display_name": "Python 3",
            "language": "python",
            "name": "python3",
        },
        "language_info": {
            "name": "python",
            "version": "3.11",
        },
    },
)

with open("exported_error_metrics_notebook.ipynb", "w", encoding="utf-8") as notebook_file:
    nbformat.write(notebook_export, notebook_file)

print("Notebook export example written to exported_error_metrics_notebook.ipynb")

## Conclusion

The `ErrorMetrics` class provides a compact, reusable way to evaluate both regression and classification outputs.

It demonstrates:
- mathematically correct metric implementations
- robust input validation
- numerical stability for log loss
- agreement with scikit-learn on standard metrics
- faster vectorized computation than naive loops